# Lab06 Calculations

In [1]:
import numpy as np
import pandas as pd

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

## Bài 1 - Attention scores

In [2]:
q = np.array([1, 2])
K = np.array([
    [1, 0],
    [0, 1],
    [1, 1]
])

scores_b1 = []
for i, k_i in enumerate(K):
    s_i = np.dot(q, k_i)
    scores_b1.append(s_i)
    print(f"s{i+1} = q^T k{i+1} = {q[0]}*{k_i[0]} + {q[1]}*{k_i[1]} = {s_i}")

scores_b1 = np.array(scores_b1)
print("Score vector s =", scores_b1)

s1 = q^T k1 = 1*1 + 2*0 = 1
s2 = q^T k2 = 1*0 + 2*1 = 2
s3 = q^T k3 = 1*1 + 2*1 = 3
Score vector s = [1 2 3]


Score vector thu được là s = [1, 2, 3]. Key thứ ba k3 = [1, 1] có điểm cao nhất vì nó cùng hướng với cả hai thành phần của query, trong khi k1 và k2 chỉ khớp được với một chiều. Riêng k2 có điểm cao hơn k1 vì chiều thứ hai của query có giá trị 2, lớn gấp đôi chiều thứ nhất, nên key nào khớp ở chiều thứ hai sẽ được lợi hơn.

## Bài 2 - Softmax

In [3]:
s_b2 = np.array([1.0, 2.0, 3.0])

exp_s = np.exp(s_b2)
alpha_b2 = exp_s / np.sum(exp_s)

b2_df = pd.DataFrame({
    "s_i": s_b2,
    "exp(s_i)": np.round(exp_s, 4),
    "alpha_i": np.round(alpha_b2, 4)
}, index=["token 1", "token 2", "token 3"])
display(b2_df)
print("Tong exp(s_j) =", round(float(np.sum(exp_s)), 4))
print("Tong alpha_i =", round(float(np.sum(alpha_b2)), 6))

,s_i,exp(s_i),alpha_i
token 1,1.0,2.7183,0.0900
token 2,2.0,7.3891,0.2447
token 3,3.0,20.0855,0.6652


Tong exp(s_j) = 30.1929
Tong alpha_i = 1.0


Sau softmax ta được khoảng 0.09, 0.245 và 0.665, cộng lại đúng bằng 1 nên có thể xem như một phân phối xác suất. Phần tử có score lớn nhất nhận weight lớn nhất vì hàm mũ là hàm đồng biến, score lớn hơn thì exp lớn hơn, mà mẫu số chung cho cả ba phần tử nên thứ tự được giữ nguyên.

Một điểm đáng để ý là score chỉ tăng đều 1 đơn vị mỗi bước nhưng weight lại tăng gần gấp 2.7 lần mỗi bước. Như vậy softmax không chỉ giữ thứ tự mà còn khuếch đại khoảng cách giữa các score, token thứ ba chiếm tới hai phần ba tổng trọng số dù score của nó chỉ hơn token thứ hai đúng 1 đơn vị.

## Bài 3 - Context vector

In [4]:
alpha_b3 = np.array([0.1, 0.2, 0.7])
V_b3 = np.array([
    [1, 0],
    [0, 1],
    [2, 2]
])

contributions = alpha_b3[:, None] * V_b3
c_b3 = contributions.sum(axis=0)

b3_df = pd.DataFrame({
    "alpha_i": alpha_b3,
    "v_i": [list(v) for v in V_b3],
    "alpha_i * v_i": [list(np.round(c, 2)) for c in contributions],
    "Do lon dong gop": np.round(np.linalg.norm(contributions, axis=1), 4)
}, index=["v1", "v2", "v3"])
display(b3_df)
print("Context vector c =", np.round(c_b3, 4))

,alpha_i,v_i,alpha_i * v_i,Do lon dong gop
v1,0.1,"[1, 0]","[0.1, 0.0]",0.1000
v2,0.2,"[0, 1]","[0.0, 0.2]",0.2000
v3,0.7,"[2, 2]","[1.4, 1.4]",1.9799


Context vector c = [1.5 1.6]


Context vector là c = [1.5, 1.6]. Value v3 đóng góp nhiều nhất, phần của nó là [1.4, 1.4], gần như chiếm trọn context vector. Lý do là v3 vừa có weight lớn nhất 0.7 vừa có độ lớn lớn nhất trong ba value. Ở đây cần phân biệt hai thứ: weight cho biết mô hình chú ý vào đâu, còn đóng góp thực sự vào context phụ thuộc cả weight lẫn độ lớn của value. Nếu v3 có độ lớn nhỏ thì dù weight 0.7 nó cũng chưa chắc chiếm ưu thế như vậy.

## Bài 4 - Toàn bộ attention computation

In [5]:
q_b4 = np.array([1, 0])
K_b4 = np.array([
    [1, 0],
    [0, 1],
    [1, 1]
])
V_b4 = np.array([
    [1, 0],
    [0, 2],
    [2, 1]
])

# Buoc 1: scores
scores_b4 = K_b4 @ q_b4
print("Buoc 1 - scores:", scores_b4)

# Buoc 2: softmax
exp_b4 = np.exp(scores_b4)
alpha_b4 = exp_b4 / np.sum(exp_b4)
print("Buoc 2 - exp(scores):", np.round(exp_b4, 4), "| tong =", round(float(exp_b4.sum()), 4))
print("Buoc 3 - attention weights:", np.round(alpha_b4, 4), "| tong =", round(float(alpha_b4.sum()), 4))

# Buoc 3: weighted sum
c_b4 = np.zeros(2)
for i in range(len(V_b4)):
    c_b4 += alpha_b4[i] * V_b4[i]
    print(f"   alpha{i+1} * v{i+1} = {alpha_b4[i]:.4f} * {V_b4[i]} = {np.round(alpha_b4[i] * V_b4[i], 4)}")
print("Buoc 4 - context vector c =", np.round(c_b4, 4))

Buoc 1 - scores: [1 0 1]
Buoc 2 - exp(scores): [2.7183 1.     2.7183] | tong = 6.4366
Buoc 3 - attention weights: [0.4223 0.1554 0.4223] | tong = 1.0
   alpha1 * v1 = 0.4223 * [1 0] = [0.4223 0.    ]
   alpha2 * v2 = 0.1554 * [0 2] = [0.     0.3107]
   alpha3 * v3 = 0.4223 * [2 1] = [0.8446 0.4223]
Buoc 4 - context vector c = [1.267 0.733]


Query q = [1, 0] chỉ quan tâm tới chiều thứ nhất nên k1 và k3 cùng được điểm 1, còn k2 nằm vuông góc với q nên điểm bằng 0. Kết quả là k1 và k3 chia đều nhau khoảng 42.2 phần trăm, k2 vẫn còn khoảng 15.5 phần trăm chứ không bị loại hẳn.

Context vector cuối cùng là khoảng [1.267, 0.733]. Chiều thứ nhất lớn hơn hẳn vì hai value được chú ý nhiều nhất là v1 và v3 đều có thành phần đầu dương. Chiều thứ hai vẫn có giá trị vì v3 có thành phần thứ hai bằng 1 và v2 dù chỉ nhận 15.5 phần trăm nhưng có thành phần thứ hai bằng 2. Bài này cho thấy rõ query chỉ quyết định trọng số, còn nội dung đưa vào context lại đến từ value.

## Bài tập Scaling

In [6]:
d_k = 4
raw_scores = np.array([4.0, 2.0, 0.0])
scaled_scores = raw_scores / np.sqrt(d_k)

w_raw = softmax(raw_scores)
w_scaled = softmax(scaled_scores)

scaling_df = pd.DataFrame({
    "QK^T": raw_scores,
    "QK^T / sqrt(4)": scaled_scores,
    "Softmax khong scaling": np.round(w_raw, 4),
    "Softmax co scaling": np.round(w_scaled, 4)
}, index=["key 1", "key 2", "key 3"])
display(scaling_df)

,QK^T,QK^T / sqrt(4),Softmax khong scaling,Softmax co scaling
key 1,4.0,2.0,0.8668,0.6652
key 2,2.0,1.0,0.1173,0.2447
key 3,0.0,0.0,0.0159,0.0900


Khi không scaling, key đầu tiên chiếm gần 87 phần trăm và key cuối chỉ còn chưa tới 2 phần trăm. Sau khi chia cho căn của d_k bằng 2, khoảng cách giữa các score co lại một nửa nên phân phối mềm hơn hẳn, key đầu còn khoảng 66.5 phần trăm và key cuối tăng lên 9 phần trăm.

Như vậy scaling không đổi thứ tự ưu tiên của các key mà chỉ làm softmax bớt tập trung. Điều này quan trọng khi d_k lớn, vì tích vô hướng có xu hướng lớn dần theo số chiều, nếu không chia thì softmax rất dễ dồn gần hết trọng số vào một key và gradient ở các key còn lại gần như bằng 0.

## Bài tập Temperature

In [7]:
s_temp = np.array([4.0, 2.0, 0.0])

temp_rows = []
for T in [0.5, 1.0, 2.0]:
    w = softmax(s_temp / T)
    ent = -np.sum(w * np.log(w))
    temp_rows.append({
        "Temperature": T,
        "Weights": np.round(w, 4),
        "Max weight": round(float(w.max()), 4),
        "Entropy": round(float(ent), 4)
    })
display(pd.DataFrame(temp_rows))

,Temperature,Weights,Max weight,Entropy
0,0.5,"[0.9817, 0.018, 0.0003]",0.9817,0.0930
1,1.0,"[0.8668, 0.1173, 0.0159]",0.8668,0.4411
2,2.0,"[0.6652, 0.2447, 0.09]",0.6652,0.8324


Bảng cho thấy đúng xu hướng mà đề bài muốn quan sát. Với T bằng 0.5, key đầu chiếm tới 98 phần trăm, entropy chỉ khoảng 0.09, gần như là chọn cứng một key. Với T bằng 2, max weight giảm còn 0.665 và entropy tăng lên 0.83, phân phối phẳng hơn. Kết quả ở T bằng 2 trùng đúng với trường hợp chia cho căn của 4 ở bài trước, nên có thể hiểu phép scaling trong attention chính là softmax với temperature bằng căn của d_k.

## Bài tập Self-attention

Đề bài không cho giá trị cụ thể của X và các ma trận W nên ở đây mình tự chọn các giá trị nhỏ để dễ tính tay, phần trên giấy dùng cùng bộ số này.

In [8]:
tokens_sa = ["I", "like", "cats"]

X_sa = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])
W_Q = np.array([
    [1.0, 0.0],
    [1.0, 1.0]
])
W_K = np.array([
    [1.0, 1.0],
    [0.0, 1.0]
])
W_V = np.array([
    [1.0, 0.0],
    [0.0, 2.0]
])

Q_sa = X_sa @ W_Q
K_sa = X_sa @ W_K
V_sa = X_sa @ W_V
print("Q =\n", Q_sa)
print("K =\n", K_sa)
print("V =\n", V_sa)

Q =
 [[1. 0.]
 [1. 1.]
 [2. 1.]]
K =
 [[1. 1.]
 [0. 1.]
 [1. 2.]]
V =
 [[1. 0.]
 [0. 2.]
 [1. 2.]]


In [9]:
d_k_sa = X_sa.shape[1]
QK_sa = Q_sa @ K_sa.T
S_sa = QK_sa / np.sqrt(d_k_sa)
A_sa = softmax(S_sa)
C_sa = A_sa @ V_sa

print("QK^T =")
display(pd.DataFrame(QK_sa, index=tokens_sa, columns=tokens_sa))
print("QK^T / sqrt(d_k) =")
display(pd.DataFrame(np.round(S_sa, 4), index=tokens_sa, columns=tokens_sa))
print("Attention matrix A =")
display(pd.DataFrame(np.round(A_sa, 4), index=tokens_sa, columns=tokens_sa))
print("Context C = A V =")
display(pd.DataFrame(np.round(C_sa, 4), index=tokens_sa, columns=["dim 1", "dim 2"]))

QK^T =


,I,like,cats
I,1.0,0.0,1.0
like,2.0,1.0,3.0
cats,3.0,1.0,4.0


QK^T / sqrt(d_k) =


,I,like,cats
I,0.7071,0.0000,0.7071
like,1.4142,0.7071,2.1213
cats,2.1213,0.7071,2.8284


Attention matrix A =


,I,like,cats
I,0.4011,0.1978,0.4011
like,0.2840,0.1400,0.5760
cats,0.3057,0.0743,0.6200


Context C = A V =


,dim 1,dim 2
I,0.8022,1.1978
like,0.8600,1.4320
cats,0.9257,1.3886


Từ ma trận attention có thể thấy mỗi hàng là một phân phối riêng, tức là mỗi token tự quyết định nó lấy thông tin từ những token nào. Token like không chú ý nhiều nhất vào chính nó mà vào cats với khoảng 57.6 phần trăm, còn cats cũng dồn 62 phần trăm vào chính nó. Token I chia đều cho chính nó và cats.

Context của mỗi token vì vậy là một tổ hợp của cả ba value chứ không còn là vector riêng của nó nữa. Đây chính là ý mà đề bài muốn nhấn mạnh: trong self-attention, một token có thể tương tác trực tiếp với mọi token khác trong cùng câu chỉ qua một bước nhân ma trận, không phải đi qua nhiều bước như hidden state của RNN. Cần lưu ý là các con số ở đây đến từ W tự chọn nên không mang ý nghĩa ngôn ngữ, chúng chỉ minh họa cách tính.

## Đối chiếu với module attention_numpy

In [10]:
from attention_numpy import scaled_dot_product_attention, self_attention

_, A_check, C_check = scaled_dot_product_attention(q_b4[None, :], K_b4, V_b4)
print("Bai 4 khi co scaling sqrt(2):", np.round(A_check[0], 4), np.round(C_check[0], 4))

_, _, _, _, A_mod, C_mod = self_attention(X_sa, W_Q, W_K, W_V)
print("Self-attention khop voi tinh tay:", np.allclose(A_mod, A_sa), np.allclose(C_mod, C_sa))

Bai 4 khi co scaling sqrt(2): [0.4011 0.1978 0.4011] [1.2033 0.7967]
Self-attention khop voi tinh tay: True True
